# Yeoubi: train a ~100M model on Kaggle

Kaggle port of `colab_train.ipynb`. It trains on the same parts and resumes from the same
checkpoint repo, so Colab and Kaggle sessions can take turns on one run as long as the
settings cell is identical on both.

- Settings > Accelerator: **GPU T4 x2**. Training runs one process per GPU (torchrun +
  DDP) with the same global batch as a single Colab T4, so the schedule is unchanged.
- Settings > Internet: **on** (needs a phone-verified account).
- Add-ons > Secrets: `HF_TOKEN`, a Hugging Face token with write access, attached to
  this notebook.
- Start it with **Save Version > Save & Run All (Commit)**. A committed run keeps going for
  up to 12 hours with the browser closed; an interactive session stops when the tab does.
  The weekly GPU quota is about 30 hours, so roughly two and a half runs a week.

In [ ]:
# SPDX-License-Identifier: BigCode-OpenRAIL-M-1.0
REPO = "https://github.com/Fury7425/Yeoubi.git"
HF_REPO = ""   # dataset repo holding the parts, such as "your-name/yeoubi-data"
CKPT_REPO = ""   # private model repo for checkpoints, such as "your-name/yeoubi-ckpt"
EPOCHS = 1    # passes over the data. Sets max_steps, which the LR and A8->A4 schedules
              # depend on, so never change it in the middle of a run.
WINDOW = 24   # parts fetched per session; one part is about 40 minutes on one T4, so 24
              # covers an 11 h run with room to spare.
KEEP = 2      # resumable checkpoints kept in CKPT_REPO, roughly 1 GB each
HOURS = 11    # Kaggle kills a session at 12 h without warning. train.py stops itself
              # after this many hours of training and writes step_*_preempted.pt; the
              # hour left covers setup and the last upload.

In [ ]:
import glob
import gzip
import json
import os
import random
import re
import shutil
import threading
import time

assert CKPT_REPO and HF_REPO, "set CKPT_REPO and HF_REPO"

# Everything lives under /tmp, not /kaggle/working: a committed run saves /kaggle/working
# as the notebook's output, and gigabytes of parts and checkpoints there would be copied
# for nothing at the end of every run.
WORK = "/tmp"
if not os.path.isdir(f"{WORK}/Yeoubi"):
    !git clone --depth 1 {REPO} {WORK}/Yeoubi
%cd {WORK}/Yeoubi
!pip -q install -r requirements.txt
!nvidia-smi -L

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: the checkpoint repo calls below will fail")

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

api = HfApi()
api.create_repo(CKPT_REPO, private=True, exist_ok=True)
LOCAL_CKPT = f"{WORK}/ckpt"
os.makedirs(LOCAL_CKPT, exist_ok=True)

In [ ]:
def step_of(path):
    match = re.search(r"step_(\d+)", os.path.basename(path))
    return int(match.group(1)) if match else -1


def is_resumable(name):
    # The pre-A4 snapshot and final.pt are kept for reference, never resumed from or pruned.
    return name.startswith("step_") and name.endswith(".pt") and "pre_a4" not in name


def gunzip(name, target, seed=None):
    # train.py's loader dispatches on .jsonl / .pt and does not read .gz itself.
    with gzip.open(hf_hub_download(HF_REPO, name, repo_type="dataset"), "rb") as source:
        lines = source.readlines()   # one part fits in RAM by design
    if seed is not None:
        # A part holds documents in the order the builder received them. Seeded, so a
        # resumed session reads the part in the same order.
        random.Random(seed).shuffle(lines)
    with open(target, "wb") as output:
        output.writelines(lines)


newest = max(filter(is_resumable, api.list_repo_files(CKPT_REPO)), key=step_of, default="")
resume = hf_hub_download(CKPT_REPO, newest) if newest else ""
step = step_of(newest) if newest else 0

with open(hf_hub_download(HF_REPO, "meta.json", repo_type="dataset"), encoding="utf-8") as handle:
    meta = json.load(handle)
parts, steps_per_part = meta["parts"], meta["steps_per_part"]
# scripts/pack_for_colab.py writes no total_steps.
max_steps = meta.get("total_steps", parts * steps_per_part) * EPOCHS

# The loader position is not checkpointed, so the checkpoint step picks the first part.
first = (step // steps_per_part) % parts
window_parts = [(first + index) % parts for index in range(min(WINDOW, parts))]
# Fills the local cache 8 files at a time; the gunzip() calls below then find every
# file already there.
snapshot_download(
    HF_REPO,
    repo_type="dataset",
    max_workers=8,
    allow_patterns=[f"parts/part_{part:05d}.jsonl.gz" for part in window_parts] + ["val.jsonl.gz"],
)
shutil.rmtree(f"{WORK}/train", ignore_errors=True)
os.makedirs(f"{WORK}/train")
os.makedirs(f"{WORK}/val", exist_ok=True)
for index, part in enumerate(window_parts):
    # Named by position, not part number: the loader sorts paths, and a window that
    # wraps past the last part must still be read in window order.
    gunzip(f"parts/part_{part:05d}.jsonl.gz", f"{WORK}/train/{index:03d}.jsonl", seed=part)
gunzip("val.jsonl.gz", f"{WORK}/val/val.jsonl")
print(f"resume={newest or 'none'} step={step}/{max_steps} parts {window_parts[0]}..{window_parts[-1]}")

In [ ]:
_sizes = {}
_sync_lock = threading.Lock()   # the background loop and the final sync must not overlap


def sync_checkpoints(stable_only=True):
    """Upload finished local checkpoints to CKPT_REPO and drop old ones there."""
    with _sync_lock:
        uploaded = False
        for path in sorted(glob.glob(f"{LOCAL_CKPT}/*.pt"), key=step_of):
            size = os.path.getsize(path)
            # torch.save writes in place, so a file is only known complete once its
            # size has stopped changing between two polls.
            if stable_only and _sizes.get(path) != size:
                _sizes[path] = size
                continue
            api.upload_file(
                path_or_fileobj=path, path_in_repo=os.path.basename(path), repo_id=CKPT_REPO
            )
            os.remove(path)   # only after the upload committed
            uploaded = True
        if not uploaded:
            return
        resumable = sorted(filter(is_resumable, api.list_repo_files(CKPT_REPO)), key=step_of)
        for old in resumable[:-KEEP]:
            api.delete_file(old, CKPT_REPO)
        if len(resumable) > KEEP:
            # A deleted file stays in the repo history and keeps counting toward the
            # account's storage; squashing the history is what releases it.
            api.super_squash_history(CKPT_REPO)
            # Squashing drops old files from the history, but the Hub keeps their stored
            # bytes, still counted toward the storage limit, until they are deleted by
            # hash: 89 of them (78 GB) filled the account at step 142000.
            live = {f.lfs.sha256 for f in api.list_repo_tree(CKPT_REPO, expand=True)
                    if getattr(f, "lfs", None)}
            dead = [f for f in api.list_lfs_files(CKPT_REPO) if f.file_oid not in live]
            if dead:
                api.permanently_delete_lfs_files(CKPT_REPO, dead)


def _sync_loop():
    while True:
        time.sleep(60)
        try:
            sync_checkpoints()
        except Exception as error:   # a network hiccup must not end the syncing
            print("checkpoint sync failed, will retry:", error)


if "_sync_thread" not in globals():
    _sync_thread = threading.Thread(target=_sync_loop, daemon=True)
    _sync_thread.start()

In [ ]:
import torch

# Flags match colab_train.ipynb; see the notes there for why each one is set. They must
# stay identical across sessions of one run, whichever platform runs the session.
# meta.json's step counts assume 16 chunks per optimizer step. batch_size and
# grad_accum_steps are per GPU, so the accumulation shrinks as GPUs are added.
GPUS = torch.cuda.device_count()
assert GPUS in (1, 2, 4), f"16 chunks per step do not split evenly over {GPUS} GPUs"
ACCUM = 4 // GPUS
!PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node {GPUS} train.py \
    --train_data "{WORK}/train/*.jsonl" --val_data "{WORK}/val/*.jsonl" \
    --output_dir {LOCAL_CKPT} --resume_from "{resume}" --max_steps {max_steps} \
    --max_seq_len 4096 --batch_size 4 --grad_accum_steps {ACCUM} \
    --hidden_dim 768 --n_heads 12 --ffn_dim 2048 \
    --bf16 false --compile_model false --gradient_checkpointing false \
    --save_every 500 --muon_weight_decay 0.005 --max_runtime_hours {HOURS} \
    --num_workers 2 --wandb_project ""

sync_checkpoints(stable_only=False)   # train.py has exited, so every file is complete